# W4-T3 — CM04: the held-out Tortoise attack

The only attack in this project that **no model is ever allowed to see**. CM01
(XTTS) and CM02 (RVC) are both train-pool, both seen by S3. Without CM04 the claim
"these results are not shortcut artefacts" has no held-out-tool evidence behind it —
a detector could score near-zero on CM01/CM02 by learning those two generators and
we would not be able to tell.

**Two firewalls, both mechanical, both already enforced in the committed job table:**

1. `tool == tortoise` on every row, and `tests/test_splits.py` fails the build if a
   Tortoise clip ever appears in a training manifest.
2. `pool == eval` on every row — 15 eval-pool speakers only, zero overlap with the
   25 train-pool speakers CM01/CM02 cloned.

The job table (`data/manifests/heldout_generation_jobs.csv`, 500 rows) is committed
**before** this notebook runs, so what gets generated is decided in git and audited
in review, not chosen on a GPU at 2am.

---

## Before you run

1. **Settings → Accelerator → GPU T4 x2. Internet → On.**
2. **+ Add Input** → your MUCS 2021 dataset (the notebook links it under a `raw/`
   anchor so `src.utils.paths.resolve()` can rebase the manifest — same trick as the
   CM02 notebook, P-020 defect 1).
3. **+ Add Input** → the private dataset holding `mentor_signoff*.pdf`. The ethics
   gate has no override; a fresh clone never carries the signed note.

Tortoise is **slow** — far slower than XTTS. Budget the session accordingly and use
`LIMIT` for a smoke run before committing to all 500.

## 1. Configuration

In [ ]:
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH    = "main"

# Smoke first: 5 clips, confirm the chain runs end to end, then set LIMIT = None.
LIMIT = 5

JOBS_CSV = "data/manifests/heldout_generation_jobs.csv"
PRESET   = "fast"   # tortoise quality preset: ultra_fast | fast | standard | high_quality

## 2. Clone the repo (public — no token needed)

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"


def sh(cmd, cwd=None, check=True):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout[-3000:])
    if r.returncode != 0:
        if r.stderr.strip():
            print(r.stderr[-3000:])
        if check:
            raise SystemExit("command failed with exit %d" % r.returncode)
    return r


if REPO.exists():
    shutil.rmtree(REPO)
sh(["git", "clone", "--branch", BRANCH, "--depth", "1", "https://%s" % REPO_HOST, str(REPO)])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh("git log --oneline -3")

## 3. Ethics gate

No override exists (`src/data/ethics_gate.py`). This runs before any model is
downloaded, so a blocked session costs seconds rather than a Tortoise install.

In [ ]:
signoff = glob.glob("/kaggle/input/**/mentor_signoff*.pdf", recursive=True)
if not signoff:
    raise SystemExit(
        "no mentor_signoff*.pdf under /kaggle/input -- attach the ethics dataset (step 3 above)"
    )
Path("docs/ethics").mkdir(parents=True, exist_ok=True)
shutil.copy2(signoff[0], Path("docs/ethics") / Path(signoff[0]).name)

from src.data.ethics_gate import signoff_status

status = signoff_status()
print(status.describe())
if not status.signed:
    raise SystemExit("ethics gate closed")

## 4. Link MUCS under a `raw/` anchor and cut the reference clips

`/kaggle/input` is read-only and the upload does not preserve the `raw/` anchor that
`paths.resolve()` rebases on, so the corpus is symlinked into `/kaggle/working`
(P-020, defect 1). References are cut here rather than shipped, so the repo carries
no audio.

In [ ]:
mucs = glob.glob("/kaggle/input/**/mucs2021", recursive=True)
if not mucs:
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("  ", p)
    raise SystemExit("mucs2021 not found under /kaggle/input")

DATA_ROOT = WORK / "dfdata"
(DATA_ROOT / "raw").mkdir(parents=True, exist_ok=True)
link = DATA_ROOT / "raw" / "mucs2021"
if not link.exists():
    os.symlink(mucs[0], link, target_is_directory=True)
os.environ["DATA_ROOT"] = str(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

import pandas as pd

from src.data.corpora import index_mucs
from src.data.pilot_jobs import choose_references
from src.utils.audio_utils import TARGET_SR, save_wav
from src.data.corpora import load_clip

jobs = pd.read_csv(JOBS_CSV)
pools = pd.read_csv("data/manifests/speaker_pools.csv")
print("jobs:", len(jobs), "| tools:", jobs.tool.unique(), "| pools:", jobs["pool"].unique())
assert set(jobs.tool) == {"tortoise"} and set(jobs["pool"]) == {"eval"}, "firewall broken"

index = index_mucs(str(link), split="train")
refs_dir = WORK / "cm04_pack" / "refs"
refs_dir.mkdir(parents=True, exist_ok=True)
references = choose_references(index, pools, pool="eval")
references = references.set_index(references["speaker"].astype(str))
for speaker in sorted(set(jobs["speaker"].astype(str))):
    target = refs_dir / f"{speaker}.wav"
    if target.exists():
        continue
    audio, sr = load_clip(references.loc[speaker].to_dict(), target_sr=TARGET_SR)
    save_wav(str(target), audio, sr)
print("references ready:", len(list(refs_dir.glob("*.wav"))))

## 5. Install Tortoise

Pinned deliberately: the held-out attack must stay reproducible, and a silent
upstream change to the generator would make CM04 a different attack than the one
the results doc describes.

In [ ]:
sh([sys.executable, "-m", "pip", "install", "-q", "tortoise-tts==3.0.0"], check=False)

import torch

print("torch     ", torch.__version__)
print("cuda avail", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("no GPU -- set Accelerator to GPU T4 x2")

from tortoise.api import TextToSpeech  # noqa: F401

print("tortoise imports OK")

## 6. Generate

`src.data.heldout_tts` holds the pipeline logic — the firewall assertions, the
metadata records and the resumable batch loop. This cell only wires paths to it
(team rules, section 9: notebooks carry no pipeline logic).

In [ ]:
from src.data.heldout_tts import build_heldout_jobs, generate_heldout_batch, load_tortoise

speaker_refs = {s: str(refs_dir / f"{s}.wav") for s in sorted(set(jobs["speaker"].astype(str)))}
transcripts = list(zip(jobs["speaker"].astype(str), jobs["transcript"].astype(str)))

OUT = WORK / "cm04_outputs"
clone_jobs = build_heldout_jobs(speaker_refs, transcripts, str(OUT), n_target=LIMIT)
print("jobs to run:", len(clone_jobs))

model = load_tortoise()
metadata = str(OUT / "generation_metadata.jsonl")
written = generate_heldout_batch(clone_jobs, model, metadata_path=metadata)
print("generated", len(written), "clip(s) of", len(clone_jobs), "requested")

## 7. QA screen, portable metadata, checksums

Same three steps CM02 went through, and for the same reason: the failure rate is a
number the datasheet has to state, machine-local paths must never enter git, and an
archive nobody can verify is not an archive.

In [ ]:
import json

from src.data.generation_qa import screen, summarise
from src.data.rvc_report import portable_records

report = screen(jobs.head(len(written)), str(OUT))
qa = summarise(report)
print(json.dumps(qa, indent=2)[:800])
Path("docs/qa").mkdir(parents=True, exist_ok=True)
report.to_csv("docs/qa/heldout_generation_qa.csv", index=False)

records = [json.loads(line) for line in Path(metadata).read_text(encoding="utf-8").splitlines() if line.strip()]
portable = portable_records(records, root=str(DATA_ROOT))
out_meta = WORK / "heldout_generation_metadata.jsonl"
out_meta.write_text(
    "\n".join(json.dumps(r, ensure_ascii=False, sort_keys=True) for r in portable) + "\n",
    encoding="utf-8",
)
text = out_meta.read_text(encoding="utf-8")
for bad in ("/kaggle/", "C:/", "C:\\"):
    assert bad not in text, f"absolute path leaked: {bad}"
print("portable metadata written:", out_meta, len(portable), "records")

## 8. Collect

Download `heldout_generation_metadata.jsonl` and the QA report from the Output tab
and commit them. **The audio is never committed** — zip it to a private Kaggle
dataset, exactly as CM01 and CM02 were, then verify with
`python -m src.data.rvc_archive --root <archive> --no-models --metadata outputs/heldout_generation_metadata.jsonl`.

In [ ]:
shutil.make_archive(str(WORK / "tortoise_cm04_clips"), "zip", str(OUT))
print("archive:", (WORK / "tortoise_cm04_clips.zip").stat().st_size / 1e6, "MB")
print()
print("Next, on a machine with the repo:")
print("  1. commit outputs/heldout_generation_metadata.jsonl + docs/qa/heldout_generation_qa.md")
print("  2. upload the zip as a PRIVATE kaggle dataset")
print("  3. update the CM04 row in docs/attack_taxonomy.md and docs/datasheet.md with actuals")
print("  4. verify: python -m src.data.rvc_archive --root <archive> --no-models \\")
print("       --metadata outputs/heldout_generation_metadata.jsonl")